# Matrixize `rMATS` Data

## Purpose: 

Convert `rMATS` results files into matrices

## Packages and Options

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, os 
import numpy as np 
import sys

## Literals

In [2]:
cell_lines = ["HepG2", "K562"]
splice_types = ["A3SS", "A5SS", "SE", "MXE", "RI"]

# directory name to pull either batch-corrected or not batch-corrected files 
norm_or_not_pattern = "MATS_output"

# choose whether you want skipping and junction counts to be summated or not
get_total_counts = True 

# whether you want inclevel vs IJC/SJC counts
get_inclevel=True

# order of samples if junction and skipping counts are SUMMATED 
summated_sample_ordering = ["KO_Sample_1", "KO_Sample_2", "CTRL_Sample_1", "CTRL_Sample_2"]
# order of samples if junction and skipping counts are NOT SUMMATED
non_summated_sample_ordering = ["KO_Sample_1_IJC", "KO_Sample_2_IJC", "KO_Sample_1_SJC", "KO_Sample_2_SJC", "CTRL_Sample_1_IJC", "CTRL_Sample_2_IJC", "CTRL_Sample_1_SJC", "CTRL_Sample_2_SJC"]



## Matrixization Algorithm

In [3]:
def matrixize_rmats_table(file = None, get_total_counts=None, inclevel=None):
    """
    Takes all rMATS files and extracts genomic features from them to create a matrix using junction and skipping counts. 
    
    Parameters:
        files (list): list of file paths where the rMATS files can be found (default = None)
        get_total_counts (bool): whether to sum junction and skipping counts or keep them separate (default = None)   
        inclevel (bool): if True, take the inclusion level and if False use the SJC/IJC counts 
    
    Returns: 
        matrix (pandas.DataFrame): a Pandas DataFrame where features are columns and rows are samples
        
    """
    
    assert file!=None and get_total_counts!=None and inclevel!=None
    
     # need to extract rbp and cell line from path 
    rbp_cell_line = None

    for path_part in file.split("/"): 
        # save the entire line that has the rbp-batch-cell_line nomenclature
        if "HepG2" in path_part or "K562" in path_part: 
            rbp_cell_line = path_part 

    assert rbp_cell_line != None, rbp_cell_line
    
    # dictionary where genomic position feature is key and value is sub-dict 
    # sub-dict has key for sample and value of that is the counts 
    # NOTE: if "get_total_counts=False", you will have double the sampls since "SJC" and "IJC" will be kept separate. 
    matrix_dict = {}

    # load as dataframe 
    tmp_df = pd.read_csv(file,sep="\t")

    # get index position of concatenation start string 
    # we are making strings from the exact chromosome, strand and genomic positions 
    # to do so, we need to concatenate all rows after "chr" column until the "ID.1" column 
    # this is a known assumption that all the chromosome, strand, and genomic position information 
    # is between the "chr" column upto and excluding "ID.1" column
    concatenation_start = tmp_df.columns.tolist().index("chr")
    concatenation_end = tmp_df.columns.tolist().index("ID.1")

    # take all columns to make feature name 
    # convert them to strings 
    # concatenate each column row-wise with "_" as delimiter
    # (e.g. chr17_-_62496792_62497000_62496792_62496891_62498127_62498187)
    tmp_df["Feature"] = tmp_df.iloc[
        :,concatenation_start:concatenation_end
    ].astype("str").apply(
        lambda x: '_'.join(x.values.tolist()), axis=1
    )
    
    # get inclevel columns
    if inclevel: 
        count_columns = ["IncLevel1", "IncLevel2"]
    # get IJC/SJC counts columns
    elif not inclevel: 
        count_columns = ["IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2"]
        
    # take all the counts that are comma separated per column 
    # and combine them into one column that is entirely comma-separated 
    # e.g. 780,750	758,759	260,253	543,571	 becomes 780,750,758,759,260,253,543,571
    tmp_df["Counts"] = tmp_df[count_columns].astype(str).apply(
        lambda x: ",".join(x.values.tolist()),axis=1
    )

    # subset to the columns involving features and counts 
    tmp_df = tmp_df[["Feature", "Counts"]]

    # for each feature we are now going to sum up the skipping and inclusion junction counts
    # there are 8 numbers corresponding to 4 samples
    # we are using the sample count summation indices dictionary that tells us which indices to pull from list
    for row in tmp_df.itertuples():
        
        feature = row[1]
        matrix_dict[feature] = {}
        
        numbers = row[2]
        numbers = numbers.split(',') 
        
        # get inclevel values 
        if inclevel: 
            for sample_name, ratio in zip(summated_sample_ordering, numbers): 
                
                sample_name = "{}-{}".format(rbp_cell_line, sample_name)
                
                # if NA then save as Numpy NaN
                if ratio =="NA": 
                    matrix_dict[feature][sample_name] = np.nan
                # just save the inclevel value
                else: 
                    matrix_dict[feature][sample_name] = ratio
                                
        # get actual counts (i.e. SJC/IJC)
        elif not inclevel: 

            # for each feature, we need to sum the skipping and inclusion counts 
            # 4 samples so 4 features
            if get_total_counts: 
                # get list of length 4  
                summations = summate_counts(numbers)

                assert len(summations)==4 and len(summated_sample_ordering)==4

                # assign each value in their respective order with correct sample naming
                for sample_name, value in zip(summated_sample_ordering, summations): 

                    # include RBP, batch, and cell line info in the sample name 
                    sample_name = "{}-{}".format(rbp_cell_line, sample_name)

                    matrix_dict[feature][sample_name] = value    


            # OR ELSE for each feature, separate skipping and inculsion counts 
            # 4 samples * (SJC/IJC) = 8 features 
            elif not get_total_counts:

                assert len(numbers)==8 and len(non_summated_sample_ordering)==8

                # assign each value in correct order 
                for sample_name, value in zip(non_summated_sample_ordering, numbers): 

                    # include RBP, batch, and cell line info in the sample name 
                    sample_name = "{}-{}".format(rbp_cell_line, sample_name)

                    matrix_dict[feature][sample_name] = value   
    
    # return dataframe where features are the index and columns are samples 
    return pd.DataFrame.from_dict(
        matrix_dict, 
        orient="index"
    )
    

def summate_counts(numbers): 
    """
    Parameters: 
        numbers (list): list of numbers to be summated in particular order
    
    Returns: 
        summation_list (list): list of numbers that have come as a result of correct summations
    
    """
        
    # list to save the summations 
    summation_list = []
    
    # which list indices should be pulled out for each sample
    # when summating and finding out counts per sample per feature 
    sample_count_summation_indices = {
        "KO_1": [0,2],
        "KO_2": [1,3],
        "CTRL_1": [4,6],
        "CTRL_2": [5,7],
    }
                
    # to get counts for each sample
    # pull the correct indices corresponding to junction and skipping counts for that sample 
    # sum that up and save those results in dictionary
    for sample in sample_count_summation_indices: 
        summation_list.append(
            sum(
                [int(numbers[index]) for index in sample_count_summation_indices[sample] ]
            )
        
        )
        
    return summation_list
        
        

## Validation of Matrix Creation Algorithm

In [4]:
random_file = '/scratch/jve4pt/ABCF1-BGHLV30-HepG2/MATS_output/A3SS.MATS.JunctionCountOnly_corrected_pval_yogi_february_2024.tsv'

original_df = pd.read_csv(random_file, sep="\t")

original_df.iloc[111:114, :]

,ID,GeneID,geneSymbol,chr,strand,longExonStart_0base,longExonEnd,shortES,shortEE,flankingES,flankingEE,ID.1,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncFormLen,SkipFormLen,PValue,FDR,IncLevel1,IncLevel2,IncLevelDifference
111,10009,ENSG00000166313.14,APBB1,chr11,-,6422803,6423212,6422803,6422918,6423311,6423439,10009,"5,0","26,7","0,0","0,2",197,99,1.000000,1.0,"0.088,0.0","NA,0.0",0.044
112,1001,ENSG00000145041.11,VPRBP,chr3,-,51452077,51452311,51452077,51452262,51454240,51454325,1001,"118,37","0,3","25,49","0,2",147,99,0.591525,1.0,"1.0,0.893","1.0,0.943",-0.025
113,10014,ENSG00000161647.14,MPP3,chr17,-,41891564,41891729,41891564,41891695,41893405,41893447,10014,"6,2","1,0","0,1","0,0",132,99,1.000000,1.0,"0.818,1.0","NA,1.0",-0.091


In [5]:
matrixize_rmats_table(file = random_file, inclevel=True, get_total_counts=True).iloc[111:114,:]


,ABCF1-BGHLV30-HepG2-KO_Sample_1,ABCF1-BGHLV30-HepG2-KO_Sample_2,ABCF1-BGHLV30-HepG2-CTRL_Sample_1,ABCF1-BGHLV30-HepG2-CTRL_Sample_2
chr11_-_6422803_6423212_6422803_6422918_6423311_6423439,0.088,0.0,NaN,0.0
chr3_-_51452077_51452311_51452077_51452262_51454240_51454325,1.0,0.893,1.0,0.943
chr17_-_41891564_41891729_41891564_41891695_41893405_41893447,0.818,1.0,NaN,1.0


## Create All Matrices

In [6]:
all_matrices = {}

# for each cell line 
for cell_line in cell_lines: 
    
    all_matrices[cell_line] = {}
    
    # for each splice type 
    for splice_type in splice_types: 
        
        # dataframe for outer joining 
        join_df = pd.DataFrame()
        
        "{} {}".format(cell_line, splice_type)
        
        # get all files matching cell line and splice type 
        matching_files = sorted(glob.glob(
            "/scratch/jve4pt/*{}*/**/{}/{}*yogi*.tsv".format(cell_line, norm_or_not_pattern, splice_type), 
            recursive=True
        ))
        
        # for each rMATS table
        for file in matching_files: 
            
            # convert rMATS table to matrix of counts 
            matrix = matrixize_rmats_table(
                file = file, 
                get_total_counts = get_total_counts, 
                inclevel=get_inclevel
            )
            
            if get_total_counts: 
                assert len(matrix.columns)==4
            elif not get_total_counts: 
                assert len(matrix.columns)==8
                
            join_df = join_df.join(matrix, how="outer")
        
        # check that number of samples is as expected
        if get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*4
        elif not get_total_counts: 
            assert len(join_df.columns)==len(matching_files)*8
                    
        # save to dictionary 
        all_matrices[cell_line][splice_type] = join_df


'HepG2 A3SS'

'HepG2 A5SS'

'HepG2 SE'

'HepG2 MXE'

'HepG2 RI'

'K562 A3SS'

'K562 A5SS'

'K562 SE'

'K562 MXE'

'K562 RI'

## Find Duplicate Samples

#### Get preliminary dictionary with duplicated samples

In [7]:
# duplicate samples dictionary 
duplicated_dict = {}

# iterate through splice types and cell lines 
# initialize necessary subdicts
for cell_line in cell_lines: 
    duplicated_dict[cell_line] = {}
    
    for splice_type in splice_types:     
        "{} {}".format(cell_line, splice_type)

        
        # transpose so that features are columns and samples are rows 
        tmp_df = all_matrices[cell_line][splice_type].T
        tmp_df.columns.size
        
        # get the features that have no missing values and save as list 
        columns_to_keep = (tmp_df.notna().sum() / tmp_df.index.size)==1
        columns_to_keep = columns_to_keep[columns_to_keep==True].index.to_list()
        
        # subset original data to these non-NaN features
        tmp_df = tmp_df[columns_to_keep]
        tmp_df.columns.size
        
        # keep all the rows that are duplicated
        tmp_df = tmp_df[tmp_df.duplicated(keep=False)]    
        
        # group the rows by all columns and get dict of row values to samples
        # save to duplicate dict 
        duplicated_dict[cell_line][splice_type]= tmp_df.groupby(list(tmp_df)).groups
        
                

'HepG2 A3SS'

237986

720

'HepG2 A5SS'

103112

...

202061

704

'K562 RI'

31804

1527

### Validation of duplicated samples


Need to check whether each set of duplicated samples: 
* Are from the same batch
* Are the only set from that batch

In [8]:
# for each cell line and splice type 
for cell_line in cell_lines:     
    for splice_type in splice_types: 
        
        # each batch per set of duplicates saved 
        all_batches = []
        
        # for each group of duplicated samples 
        for key in duplicated_dict[cell_line][splice_type]: 
            
            # get batch identifier for each sample in list 
            # and assert that there is no more than 1 unique entry 
            batch_id = list(
                set(
                    [ID.split("-")[1] for ID in duplicated_dict[cell_line][splice_type][key].to_list()]
                )
            )
            assert len(batch_id)==1, batch_id

            # append batch id to list of batch ids for each group of duplicates 
            all_batches.append(batch_id[0])
                    
        # this makes sure that between each set of duplicated samples, no batch IDs are used twice
        # this is since the value_counts() should yield a batch ID no more than twice (1 for each control sample)
        # if so, you can say that a set of duplicated samples represents batch "X" control sample "y"
        batches_across_duplicates = pd.Series(all_batches).value_counts()
        batches_across_duplicates[batches_across_duplicates>2]


BGHLV14    4
BGHLV12    4
BGHLV20    4
Name: count, dtype: int64

BGHLV20    4
BGHLV14    4
BGHLV12    4
Name: count, dtype: int64

BGHLV12    4
BGHLV20    4
BGHLV14    4
Name: count, dtype: int64

BGHLV20    4
BGHLV12    4
BGHLV14    4
Name: count, dtype: int64

BGHLV20    4
BGHLV14    4
BGHLV12    4
Name: count, dtype: int64

LV08       4
BGKLV21    4
BGKLV29    4
BGKLV25    4
BGKLV24    4
BGKLV19    4
BGKLV13    4
Name: count, dtype: int64

BGKLV13    4
BGKLV21    4
BGKLV24    4
LV08       4
BGKLV29    4
BGKLV19    4
BGKLV25    4
Name: count, dtype: int64

BGKLV29    4
BGKLV13    4
LV08       4
BGKLV21    4
BGKLV19    4
BGKLV24    4
BGKLV25    4
Name: count, dtype: int64

LV08       4
BGKLV29    4
BGKLV24    4
BGKLV21    4
BGKLV13    4
BGKLV25    4
BGKLV19    4
Name: count, dtype: int64

BGKLV13    4
BGKLV29    4
BGKLV21    4
BGKLV24    4
BGKLV25    4
BGKLV19    4
LV08       4
Name: count, dtype: int64

**THIS MEANS THAT NOT EVERY BATCH USED 2 CONTROL SAMPLES AND HENCE, ARE NOT THE CONTROLS FOR THE ENTIRE BATCH**

Across all cell lines and splice types and for each duplicated sample group: 
* Is there the same number of them across all combinations? (There should be)
* Do they contain the same set of samples across all combinations?

In [9]:
# for each cell line 
for cell_line in cell_lines:
    # take A5SS as an example to compare against all other splice types downstream 
    # pull out all duplicated sample groups 
    comparison_dict = duplicated_dict[cell_line]["A5SS"]
    control_duplicated_sample_groups = [comparison_dict[key].to_list() for key in comparison_dict]
    
    for splice_type in splice_types: 
        
        # make sure that the number of duplicate sample groups is the same
        assert len(comparison_dict.keys()) == len(duplicated_dict[cell_line][splice_type].keys())
        
        # for each group of duplicated samples in current iteration
        for key in duplicated_dict[cell_line][splice_type]: 
            # get list of duplicate sample group
            sample_group= duplicated_dict[cell_line][splice_type][key].to_list()
            
            # this makes sure that the exact list of duplicated sample groups is found 
            # in our "control" set of duplicated sample groups which ultimately ensures 
            # that the same samples are marked as duplicates across splice types 
            assert sample_group in control_duplicated_sample_groups 

### Visual Validation that Duplicates are Actually Duplicates

Get some samples to visually look at

In [10]:
# transpose so that features are columns and samples are rows 
tmp_df = all_matrices["HepG2"]["A3SS"].T
tmp_df.columns.size

# get the features that have no missing values and save as list 
columns_to_keep = (tmp_df.notna().sum() / tmp_df.index.size)==1
columns_to_keep = columns_to_keep[columns_to_keep==True].index.to_list()

# subset original data to these non-NaN features
tmp_df = tmp_df[columns_to_keep]
tmp_df.columns.size

# keep all the rows that are duplicated
tmp_df = tmp_df[tmp_df.duplicated(keep=False)] 

# select 3 samples to compare 
tmp_df[
    tmp_df.index.isin(
        ['DDX52-BGHLV16-HepG2-CTRL_Sample_2','EEF2-BGHLV16-HepG2-CTRL_Sample_2','EFTUD2-BGHLV16-HepG2-CTRL_Sample_2']
    )
].iloc[:, 1:100]

237986

720

,chr10_+_112640990_112641293_112641032_112641293_112635723_112635828,chr10_+_120867459_120867676_120867479_120867676_120863628_120863709,chr10_+_13160887_13161040_13160905_13161040_13158266_13158340,chr10_+_27462046_27462188_27462049_27462188_27458872_27460012,chr10_+_31784704_31784767_31784707_31784767_31749965_31750166,chr10_+_46264896_46265066_46264902_46265066_46261126_46261258,chr10_+_51580879_51580968_51580891_51580968_51580555_51580696,chr10_+_75554298_75554397_75554313_75554397_75553904_75554088,chr10_+_99511132_99511219_99511147_99511219_99510087_99510227,chr10_-_112057324_112057503_112057324_112057486_112058405_112058548,chr10_-_114203272_114203364_114203272_114203352_114204927_114205408,chr10_-_116207638_116207785_116207638_116207779_116211382_116211430,chr10_-_13489265_13489319_13489265_13489316_13494538_13494658,chr10_-_3185580_3185923_3185580_3185693_3186483_3186558,chr10_-_52573616_52573822_52573616_52573798_52575765_52576039,chr10_-_70136609_70136772_70136609_70136729_70139180_70139278,chr10_-_75206249_75206334_75206249_75206331_75214169_75214247,chr10_-_88256957_88257076_88256957_88257058_88259474_88260500,chr10_-_97442857_97442952_97442857_97442937_97443278_97443386,chr11_+_111953188_111955874_111953191_111955874_111952705_111952759,chr11_+_114273549_114273637_114273553_114273637_114272419_114272582,chr11_+_114276427_114276521_114276430_114276521_114273549_114273637,chr11_+_117059423_117059542_117059444_117059542_117058343_117058415,chr11_+_118404980_118405070_118404983_118405070_118404743_118404844,chr11_+_124496708_124496946_124496799_124496946_124495566_124495799,chr11_+_126131319_126131379_126131322_126131379_126126461_126126747,chr11_+_126144821_126144916_126144825_126144916_126143230_126143349,chr11_+_57505307_57505498_57505384_57505498_57505079_57505140,chr11_+_581961_582081_581964_582081_581491_581606,chr11_+_61548428_61548517_61548431_61548517_61547693_61547762,chr11_+_62556494_62556682_62556658_62556682_62554886_62554999,chr11_+_62644255_62644351_62644258_62644351_62639048_62639141,chr11_+_64880962_64881066_64880995_64881066_64880691_64880886,chr11_+_65631271_65631372_65631274_65631372_65631070_65631192,chr11_+_65823005_65823056_65823008_65823056_65822546_65822786,chr11_+_66372624_66372887_66372809_66372887_66367959_66368020,chr11_+_66605837_66605916_66605840_66605916_66595736_66595835,chr11_+_66611426_66611510_66611430_66611510_66611229_66611332,chr11_+_67375866_67375949_67375902_67375949_67374428_67374547,chr11_+_68355394_68355492_68355412_68355492_68350510_68350597,chr11_+_758949_759057_759008_759057_755878_756002,chr11_+_8705536_8705628_8705552_8705628_8704747_8704812,chr11_+_9496096_9496180_9496099_9496180_9495487_9495571,chr11_-_111708971_111709122_111708971_111709101_111711377_111711532,chr11_-_116728525_116729415_116728525_116729235_116729980_116730320,chr11_-_117693120_117693262_117693120_117693195_117693393_117693432,chr11_-_118968564_118968753_118968564_118968635_118969112_118969197,chr11_-_18105081_18105278_18105081_18105275_18108412_18108601,chr11_-_18111721_18111863_18111721_18111837_18111980_18112040,chr11_-_2993331_2993509_2993331_2993473_2997253_2997353,chr11_-_3023199_3023404_3023199_3023283_3023770_3023830,chr11_-_47204224_47204360_47204224_47204315_47204554_47204620,chr11_-_47498397_47498511_47498397_47498508_47498848_47498977,chr11_-_47500428_47500504_47500428_47500492_47504246_47504408,chr11_-_65307956_65308102_65307956_65308085_65308341_65308425,chr11_-_65319018_65319127_65319018_65319035_65319442_65319628,chr11_-_65429649_65429818_65429649_65429676_65430296_65430389,chr11_-_67234971_67235061_67234971_67235034_67235498_67235563,chr11_-_71822202_71822332_71822202_71822327_71822457_71822542,chr11_-_9309988_9310089_9309988_9310082_9316805_9316934,chr11_-_93466515_93466585_93466515_93466563_93467790_93467826,chr11_-_93466515_93466671_93466515_93466563_93467790_93467826,chr11_-_93471274_93471665_93471274_93471606_93472402_93472497,chr11_-_95511985_95512121_95511985_95512118_9551277

## Create New Names to Label Control Samples

In [11]:
# for each cell line, store new names for duplicate samples
new_sample_names = {}

# for each cell line
for cell_line in cell_lines:
    new_sample_names[cell_line] = {}
    
    # since we showcased that the duplicate sample groups are 
    # the same across splice types, we can just use 1 of the splice types
    # iterating through all duplicate sample groups for A3SS splice type 
    for key in duplicated_dict[cell_line]["A3SS"]: 
        
        # get RBPs involved in each group of duplicate samples
        rbps_involved = [sample.split("-")[0] for sample in duplicated_dict[cell_line]["A3SS"][key].to_list()]
        rbps_involved = "_".join(rbps_involved)

        # for each sample name
        for sample in duplicated_dict[cell_line]["A3SS"][key]: 
            
            # save new sample name as combination of RBPs involved and the rest of the suffix 
            # e.g. DDX52-BGHLV16-HepG2-CTRL_Sample_2 ---> DDX52_EEF2_EFTUD2_EIF3D_FAM120A_GEMIN5_METAP2_PA2G4_PKM2_RPS19_SERBP1_TROVE2-BGHLV16-HepG2-CTRL_Sample_2
            new_sample_names[cell_line][sample] = "{}-{}".format(
                rbps_involved, 
                "-".join(
                    sample.split("-")[1:]
                )
            )


## Update Original Matrices w/ New Sample Names

In [12]:

# for each cell line and splice type 
for cell_line in cell_lines: 
    for splice_type in splice_types: 

        tmp_df = all_matrices[cell_line][splice_type].T
        
        tmp_df = tmp_df.rename(
            index= new_sample_names[cell_line]
        ) 

        tmp_df.index.size
        
        tmp_df = tmp_df[~tmp_df.index.duplicated(keep="first")]
        
        tmp_df.index.size
        
        tmp_df.T.to_csv(
            "../output/inclusion-level/{}_{}.tsv.gz".format(
                cell_line, 
                splice_type
            ), 
            compression="gzip", 
            sep="\t"
        )
        


884

490

884

490

884

...

446

804

446

804

446